# 🧪 Lab 9: Products & 구독 격리 — 토큰/요청 거버넌스 종합 테스트

## 목적
APIM **Products · Subscriptions** 로 팀별 구독을 격리하고, **두 가지 사용량 제어 레버**를
구독 단위로 적용·검증합니다. AI Gateway 가 프론티어 모델 트래픽을 **제어(control)** 하고
**관측(observability)** 하는 핵심을 실제 호출로 확인합니다.

| 레버 | 정책 | 제어 대상 | 초과 시 | 관찰 헤더 |
|---|---|---|---|---|
| **분당 토큰 레이트(TPM)** | `llm-token-limit` `tokens-per-minute` | 순간 폭증(분당 토큰) | **429** | `x-ratelimit-remaining-tokens` |
| **기간 누적 토큰 quota** | `llm-token-limit` `token-quota` | 월 토큰 총량 | **403** | `x-quota-remaining-tokens` |
| **기간 누적 요청 quota** | `quota-by-key` `calls` | 월 호출 수 | **403** | (Retry 메시지) |

## 사전 조건
- APIM 배포 완료 (Lab 1~4), Azure OpenAI 백엔드 연결
- `az login` 완료 (구독 키·정책은 이 노트북이 자동 생성/적용)
- (선택) App Insights 연동 — 실습 E 구독별 토큰 관측용

## 시나리오 구조
| 단계 | 내용 | 기대 결과 |
|---|---|---|
| 준비 | team-a / team-b Product + 구독 생성 + 정책 적용 | 구독 키 2개 |
| 실습 A | **구독 격리** — 두 키 독립 동작 | 각 200, 헤더 독립 |
| 실습 B | **TPM 429** — 분당 토큰 레이트 초과 | 429 + Retry-After |
| 실습 C | **토큰 quota 403** — 누적 토큰 총량 초과 | 403 "Token quota is exceeded" |
| 실습 D | **요청 quota 403** — 누적 호출 수 초과 | 403 "Out of call volume quota" |
| 실습 E | **관측** — App Insights 구독별 토큰 (KQL) | customMetrics 조회 |
| 정리 | Product/구독 삭제 | 원복 |

```
        Developer Portal (셀프서비스 구독)
   ┌──────────────┐         ┌──────────────┐
   │  팀 A 개발자  │         │  팀 B 개발자  │
   └──────┬───────┘         └──────┬───────┘
   Sub Key A                 Sub Key B
          │                        │
   ┌──────▼────────────────────────▼───────┐
   │   APIM  Product team-a / team-b        │
   │   ├ llm-token-limit (TPM + token-quota)│  → 429 / 403
   │   └ quota-by-key   (calls)             │  → 403
   │   counter-key = context.Subscription.Id│  (구독별 독립 카운터)
   └──────┬─────────────────────────────────┘
          ▼
     Azure OpenAI (gpt-4.1-nano)
```


In [ ]:
# ─── 환경 설정 ───
import os, time, json, subprocess, tempfile
from datetime import datetime, timezone
import requests
from dotenv import load_dotenv

load_dotenv("../../.env", override=True)

def az(args):
    """az CLI 실행 → stdout(str). 실패 시 (None, stderr)."""
    r = subprocess.run(["az"] + args, capture_output=True, text=True)
    return (r.stdout.strip(), r.stderr.strip(), r.returncode)

def az_json(args):
    out, err, rc = az(args)
    if rc != 0 or not out:
        return None
    try:
        return json.loads(out)
    except json.JSONDecodeError:
        return out

# 1) .env 우선, 없으면 현재 구독에서 자동 탐색
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID") or (az_json(["account", "show", "--query", "id", "-o", "json"]) or "")
RESOURCE_GROUP  = os.getenv("RESOURCE_GROUP", "")
APIM_NAME       = os.getenv("APIM_NAME", "")

if not (RESOURCE_GROUP and APIM_NAME):
    # 구독 내 첫 APIM 인스턴스 자동 선택
    apims = az_json(["apim", "list", "--query",
                     "[].{name:name, rg:resourceGroup, url:gatewayUrl}", "-o", "json"]) or []
    if apims:
        APIM_NAME = APIM_NAME or apims[0]["name"]
        RESOURCE_GROUP = RESOURCE_GROUP or apims[0]["rg"]

APIM_URL        = os.getenv("APIM_URL") or f"https://{APIM_NAME}.azure-api.net"
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION     = "2025-04-01-preview"
ARM_API         = "2024-06-01-preview"
APP_INSIGHTS_APP_ID = os.getenv("APP_INSIGHTS_APP_ID", "")

assert SUBSCRIPTION_ID, "❌ Azure 구독을 찾을 수 없습니다. 'az login' 후 다시 실행하세요."
assert APIM_NAME and RESOURCE_GROUP, "❌ APIM_NAME/RESOURCE_GROUP 를 찾을 수 없습니다. .env 를 확인하세요."

ARM_BASE = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
            f"/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.ApiManagement/service/{APIM_NAME}")
CHAT_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"
RUN_ID   = datetime.now(timezone.utc).strftime("%m%d%H%M%S")   # 재실행 시 카운터 격리용

def arm(method, path, body=None, query=None):
    """ARM REST 호출(az rest 래퍼). body 는 dict → 임시 JSON 파일로 전달(이스케이프 안전)."""
    url = f"{ARM_BASE}{path}?api-version={ARM_API}"
    args = ["rest", "--method", method, "--url", url]
    if query:
        args += ["--query", query, "-o", "json"]
    tmp = None
    if body is not None:
        tmp = tempfile.NamedTemporaryFile("w", suffix=".json", delete=False)
        json.dump(body, tmp); tmp.close()
        args += ["--headers", "Content-Type=application/json", "--body", f"@{tmp.name}"]
    out, err, rc = az(args)
    if tmp:
        os.unlink(tmp.name)
    return out, err, rc

print("✅ 환경 설정 완료")
print(f"   APIM      : {APIM_NAME}  ({APIM_URL})")
print(f"   RG        : {RESOURCE_GROUP}")
print(f"   Model     : {DEPLOYMENT_NAME}")
print(f"   RUN_ID    : {RUN_ID}")
print(f"   App Insights App ID : {APP_INSIGHTS_APP_ID[:8] + '...' if APP_INSIGHTS_APP_ID else '(미설정 — 실습 E 는 저장된 KQL 참고)'}")


---
## 준비: Product · 구독 · 정책 생성

> 💡 **데모용 축소 한도**: 실제 트리거를 눈으로 보기 위해 한도를 작게 설정합니다.
> 프로덕션 값(README 기준)은 아래 표의 괄호와 같습니다.

| Product | 대상 | TPM (데모/프로덕션) | 월 토큰 quota | 월 호출 수 | 구독 승인 |
|---|---|---|---|---|---|
| `team-a` | 팀 A | 5,000 / (10,000) | 1,000,000 / (10,000,000) | 100,000 | 자동 |
| `team-b` | 팀 B | 1,000 / (2,000) | 200,000 / (2,000,000) | 20,000 | 관리자 승인 |

- **통합 API 자동 탐색**: Lab 8 의 `multicloud-openai` 가 있으면 사용, 없으면 배포된 `azure-openai` 사용
- **구독 이름에 `RUN_ID`** 를 붙여 재실행 때마다 **새 구독**(= 새 카운터)으로 시작


In [ ]:
# ─── Product / API 연결 / 구독 생성 ───
API_CANDIDATES = ["multicloud-openai", "azure-openai"]
API_ID = None
for cand in API_CANDIDATES:
    out, err, rc = arm("GET", f"/apis/{cand}", query="name")
    if rc == 0 and out:
        API_ID = cand
        break
assert API_ID, "❌ 사용할 API 를 찾을 수 없습니다 (multicloud-openai / azure-openai)."
print(f"▶ 통합 API: {API_ID}")

PRODUCTS = {
    "team-a": {"display": "Team A", "approval": False, "tpm": 5000, "quota": 1000000, "calls": 100000},
    "team-b": {"display": "Team B", "approval": True,  "tpm": 1000, "quota": 200000,  "calls": 20000},
}
SUBS = {}   # product_id -> {"sid":..., "key":...}

for pid, cfg in PRODUCTS.items():
    arm("PUT", f"/products/{pid}", body={"properties": {
        "displayName": cfg["display"], "subscriptionRequired": True,
        "approvalRequired": cfg["approval"], "state": "published"}})
    arm("PUT", f"/products/{pid}/apis/{API_ID}")   # API 를 Product 에 추가
    sid = f"sub-{pid}-{RUN_ID}"
    arm("PUT", f"/subscriptions/{sid}", body={"properties": {
        "displayName": sid, "scope": f"{ARM_BASE}/products/{pid}", "state": "active"}})
    out, err, rc = arm("POST", f"/subscriptions/{sid}/listSecrets", query="primaryKey")
    key = out.strip().strip('"') if out else ""
    SUBS[pid] = {"sid": sid, "key": key}
    print(f"  ✅ {pid:<7} Product+API+구독 생성  (sid={sid}, key={'설정됨' if key else '❌'})")

print("\n⏳ 게이트웨이 전파 대기 (신규 구독/정책 반영, 약 60초)...")
time.sleep(60)
print("✅ 준비 완료")


In [ ]:
# ─── Product 정책 적용: 두 레버 + 토큰 메트릭 ───
def token_limit(counter_key, tpm, quota, period="Monthly", estimate="false"):
    return (f'<llm-token-limit counter-key="{counter_key}" '
            f'tokens-per-minute="{tpm}" token-quota="{quota}" token-quota-period="{period}" '
            f'estimate-prompt-tokens="{estimate}" '
            f'remaining-tokens-header-name="x-ratelimit-remaining-tokens" '
            f'remaining-quota-tokens-header-name="x-quota-remaining-tokens" '
            f'tokens-consumed-header-name="x-ratelimit-tokens-consumed" />')

def quota_by_key(counter_key, calls, period=2592000):
    return f'<quota-by-key counter-key="{counter_key}" calls="{calls}" renewal-period="{period}" />'

def emit_metric(provider, model):
    # Lab 8 라우팅이 없으므로 provider/model 변수를 직접 지정 → 메트릭 라벨링
    return (f'<set-variable name="provider" value="{provider}" />'
            f'<set-variable name="modelName" value="{model}" />'
            f'<llm-emit-token-metric namespace="ai-gateway-metrics">'
            f'<dimension name="Subscription ID" value="@(context.Subscription.Id)" />'
            f'<dimension name="Provider" value="@((string)context.Variables[&quot;provider&quot;])" />'
            f'<dimension name="Model" value="@((string)context.Variables[&quot;modelName&quot;])" />'
            f'<dimension name="API ID" value="@(context.Api.Id)" />'
            f'<dimension name="Client IP" value="@(context.Request.IpAddress)" />'
            f'</llm-emit-token-metric>')

def product_policy(inner):
    return ("<policies><inbound><base />" + inner +
            "</inbound><backend><base /></backend>"
            "<outbound><base /></outbound><on-error><base /></on-error></policies>")

def apply_product_policy(pid, inner, label=""):
    xml = product_policy(inner)
    out, err, rc = arm("PUT", f"/products/{pid}/policies/policy",
                       body={"properties": {"format": "rawxml", "value": xml}})
    ok = "✅" if rc == 0 else "❌"
    print(f"  {ok} {pid} 정책 적용 {label}  {'' if rc==0 else err[:160]}")
    return rc == 0

# 구독별 격리 정책(counter-key = Subscription.Id) + 토큰 메트릭
for pid, cfg in PRODUCTS.items():
    inner = (token_limit("@(context.Subscription.Id)", cfg["tpm"], cfg["quota"])
             + quota_by_key("@(context.Subscription.Id)", cfg["calls"])
             + emit_metric("azure-openai", DEPLOYMENT_NAME))
    apply_product_policy(pid, inner, label=f'(TPM={cfg["tpm"]}, quota={cfg["quota"]}, calls={cfg["calls"]})')

print("\n⏳ 정책 전파 대기 (약 40초)...")
time.sleep(40)
print("✅ 정책 적용 완료")


In [ ]:
# ─── 호출 헬퍼 ───
def call_api(key, prompt="Say hi.", max_tokens=5, timeout=30):
    """구독 키로 chat completion 호출 → (status, headers, usage, text)."""
    try:
        r = requests.post(CHAT_URL, params={"api-version": API_VERSION},
                          headers={"Content-Type": "application/json",
                                   "Ocp-Apim-Subscription-Key": key},
                          json={"messages": [{"role": "user", "content": prompt}],
                                "max_tokens": max_tokens}, timeout=timeout)
        usage = {}
        try:
            usage = r.json().get("usage", {})
        except Exception:
            pass
        return r.status_code, r.headers, usage, r.text
    except Exception as e:
        return -1, {}, {}, str(e)

def hdr(headers, name):
    return headers.get(name, "-")

# 스모크 테스트
st, h, u, _ = call_api(SUBS["team-a"]["key"])
print(f"스모크 테스트 (team-a): HTTP {st}, total_tokens={u.get('total_tokens','-')}, "
      f"remaining-TPM={hdr(h,'x-ratelimit-remaining-tokens')}, "
      f"remaining-quota={hdr(h,'x-quota-remaining-tokens')}")


---
## 실습 A: 구독 격리 (Subscription Isolation)

두 팀이 **독립된 구독 키**로 같은 API 를 호출합니다.
`counter-key = context.Subscription.Id` 이므로 **팀 A 의 사용량이 팀 B 에 영향을 주지 않습니다.**

**관찰 포인트**
- 두 키 모두 `200` 성공, `x-ratelimit-remaining-tokens` 가 **구독별로 독립 감소**
- 키 없이 호출 → `401` (구독 필수)


In [ ]:
# ─── 실습 A: 구독 격리 ───
print("═" * 60)
print(" 실습 A: 구독별 독립 카운터 검증")
print("═" * 60)
for pid in ("team-a", "team-b"):
    print(f"\n▶ {pid} (key ...{SUBS[pid]['key'][-4:]})")
    for i in (1, 2):
        st, h, u, _ = call_api(SUBS[pid]["key"])
        print(f"   [{i}] HTTP {st}  총토큰={u.get('total_tokens','-'):>3}  "
              f"남은TPM={hdr(h,'x-ratelimit-remaining-tokens'):>5}  "
              f"남은quota={hdr(h,'x-quota-remaining-tokens')}")
        time.sleep(0.4)

print("\n▶ 키 없이 호출 (구독 필수 확인)")
st, _, _, body = call_api("")
print(f"   HTTP {st}  → {'✅ 401 차단(격리 정상)' if st==401 else body[:80]}")

print("\n" + "─" * 60)
print("  핵심: 두 구독의 남은 TPM 이 서로 독립적으로 감소 → 구독 격리 ✅")
print("─" * 60)


---
## 실습 B: TPM 초과 → 429 (분당 토큰 레이트)

`llm-token-limit` 의 `tokens-per-minute` 는 **순간 폭증**을 막습니다(분당 토큰 초과 시 **429**).
team-b 정책을 **TPM=100** 으로 낮춰 짧게 폭주시키면 429 가 발생합니다.

**관찰 포인트**: `x-ratelimit-remaining-tokens` 가 0 에 수렴 → `429` + `Retry-After`


In [ ]:
# ─── 실습 B: TPM 429 ───
# team-b 정책을 TPM 전용 축소 한도로 교체(시나리오별 literal counter-key 로 결정적 트리거)
ck_b = f"lab9-{RUN_ID}-b-tpm"
apply_product_policy("team-b",
    token_limit(ck_b, tpm=100, quota=100000000) + quota_by_key(ck_b + "-calls", 100000),
    label="(TPM=100 전용)")
print("⏳ 정책 전파 대기 (약 40초)...\n"); time.sleep(40)

print("═" * 60); print(" 실습 B: 분당 토큰(TPM) 초과 → 429"); print("═" * 60)
key = SUBS["team-b"]["key"]; got_429 = False
for i in range(1, 21):
    st, h, u, body = call_api(key)
    rem = hdr(h, "x-ratelimit-remaining-tokens")
    if st == 429:
        got_429 = True
        print(f"  [{i:>2}] 🚫 429 Rate Limit!  Retry-After={h.get('Retry-After','-')}s  남은TPM={rem}")
        print(f"       body: {body[:120]}")
        break
    print(f"  [{i:>2}] HTTP {st}  남은TPM={rem}")
    time.sleep(0.2)
print("\n" + ("  → ✅ 429 트리거 성공 (TPM 레이트 제어)" if got_429
             else "  → ⚠️ 429 미발생 — TPM 을 더 낮추거나 호출을 늘리세요"))


---
## 실습 C: 누적 토큰 quota 초과 → 403 (월 토큰 예산)

`token-quota` + `token-quota-period` 는 **기간 누적 토큰 총량**을 제어합니다(초과 시 **403**).
team-b 정책을 **token-quota=40 (Hourly)** 로 낮춰 몇 번의 호출로 소진시킵니다.

**관찰 포인트**: `x-quota-remaining-tokens` 40→0 감소 → `403 "Token quota is exceeded"`


In [ ]:
# ─── 실습 C: 토큰 quota 403 ───
ck_c = f"lab9-{RUN_ID}-c-tq"
apply_product_policy("team-b",
    token_limit(ck_c, tpm=100000, quota=40, period="Hourly") + quota_by_key(ck_c + "-calls", 100000),
    label="(token-quota=40 전용)")
print("⏳ 정책 전파 대기 (약 40초)...\n"); time.sleep(40)

print("═" * 60); print(" 실습 C: 누적 토큰 quota 초과 → 403"); print("═" * 60)
key = SUBS["team-b"]["key"]; got_403 = False
for i in range(1, 8):
    st, h, u, body = call_api(key)
    remq = hdr(h, "x-quota-remaining-tokens")
    if st == 403:
        got_403 = True
        print(f"  [{i}] 🚫 403  남은quota={remq}")
        print(f"      body: {body[:140]}")
        break
    print(f"  [{i}] HTTP {st}  소비={u.get('total_tokens','-')}  남은quota={remq}")
    time.sleep(0.4)
print("\n" + ("  → ✅ 403 트리거 성공 (누적 토큰 총량 제어)" if got_403
             else "  → ⚠️ 403 미발생 — quota 를 더 낮추세요"))


---
## 실습 D: 누적 요청 quota 초과 → 403 (월 호출 수)

`quota-by-key` 는 **토큰이 아니라 호출 수(calls)** 를 집계합니다(초과 시 **403**).
team-b 정책을 **calls=3** 으로 낮춰 4번째 호출에서 차단됩니다.

**관찰 포인트**: 3회 성공 후 `403 "Out of call volume quota"`
> 💡 토큰 레버(실습 C)와 **독립적** — 요청 수 기준이므로 요청당 토큰이 적어도 호출 수로 차단됩니다.


In [ ]:
# ─── 실습 D: 요청(호출 수) quota 403 ───
ck_d = f"lab9-{RUN_ID}-d-calls"
apply_product_policy("team-b",
    token_limit(ck_d, tpm=100000, quota=100000000) + quota_by_key(ck_d, calls=3, period=3600),
    label="(calls=3 전용)")
print("⏳ 정책 전파 대기 (약 40초)...\n"); time.sleep(40)

print("═" * 60); print(" 실습 D: 누적 요청 수(calls) 초과 → 403"); print("═" * 60)
key = SUBS["team-b"]["key"]; got_403 = False
for i in range(1, 6):
    st, h, u, body = call_api(key)
    if st == 403:
        got_403 = True
        print(f"  [{i}] 🚫 403 Out of call volume quota")
        print(f"      body: {body[:140]}")
        break
    print(f"  [{i}] HTTP {st}  (호출 성공 {i}/3)")
    time.sleep(0.4)
print("\n" + ("  → ✅ 403 트리거 성공 (요청 수 제어)" if got_403
             else "  → ⚠️ 403 미발생 — calls 를 더 낮추세요"))


---
## 실습 E: 관측 — App Insights 구독별 토큰 (KQL)

`llm-emit-token-metric` 정책이 **구독별(`Subscription ID`) 토큰**을 App Insights `customMetrics`
(namespace `ai-gateway-metrics`)로 보냅니다. 준비 단계에서 이미 적용했으므로,
실습 A~D 의 트래픽이 메트릭으로 집계됩니다.

> ⚠️ **사전 활성화 필수(1회):** `emit-metric` 계열 정책은 아래 두 가지가 켜져 있어야 실제로 메트릭을 전송합니다.
> 하나라도 빠지면 **customMetrics 가 항상 비어 있습니다.** (→ 상세: [Lab 6 · 2단계 커스텀 메트릭 사전 활성화](../lab06-monitoring/README.md))
> 1. App Insights → *Usage and estimated costs* → **Custom metrics (Preview)** → **With dimensions**
> 2. APIM Diagnostics 에 **`metrics: true`** (아래 셀이 자동 확인/안내)
>
> ⏳ **수집 지연**: 활성화 후에도 `customMetrics` 는 보통 **5~10분** 뒤 조회됩니다. 방금 실행했다면 빈 결과일 수 있습니다.


In [ ]:
# ─── 실습 E: App Insights REST 로 구독별 토큰 조회 (az 토큰 인증) ───
def get_az_token():
    out, err, rc = az(["account", "get-access-token", "--resource",
                       "https://api.applicationinsights.io", "--query", "accessToken", "-o", "tsv"])
    return out.strip() if rc == 0 else ""

def query_app_insights(kql):
    if not APP_INSIGHTS_APP_ID:
        return None
    token = get_az_token()
    if not token:
        return None
    r = requests.post(f"https://api.applicationinsights.io/v1/apps/{APP_INSIGHTS_APP_ID}/query",
                      headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                      json={"query": kql}, timeout=30)
    if r.status_code != 200:
        print(f"⚠️ 조회 실패 HTTP {r.status_code}: {r.text[:160]}"); return None
    return r.json()

def print_table(result):
    if not result or not result.get("tables") or not result["tables"][0]["rows"]:
        print("  결과 없음 (데이터 수집 지연 5~10분 — 잠시 후 재실행하세요)"); return
    t = result["tables"][0]; cols = [c["name"] for c in t["columns"]]; rows = t["rows"]
    w = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0)) for i, c in enumerate(cols)]
    print("  " + "  ".join(f"{c:<{w[i]}}" for i, c in enumerate(cols)))
    print("  " + "  ".join("-" * w[i] for i in range(len(cols))))
    for r in rows:
        print("  " + "  ".join(f"{str(v):<{w[i]}}" for i, v in enumerate(r)))

KQL_PER_SUB = """
customMetrics
| where timestamp > ago(1h)
| where name in ("Total Tokens", "Prompt Tokens", "Completion Tokens")
| extend sub = tostring(customDimensions["Subscription ID"]),
         provider = tostring(customDimensions["Provider"]),
         model = tostring(customDimensions["Model"])
| summarize totalTokens = sumif(value, name=="Total Tokens"),
            promptTokens = sumif(value, name=="Prompt Tokens"),
            completionTokens = sumif(value, name=="Completion Tokens"),
            calls = dcount(timestamp)
    by sub, provider, model
| order by totalTokens desc
"""

print("═" * 60); print(" 실습 E: 구독별 토큰 사용량 (App Insights customMetrics)"); print("═" * 60)

# 사전 활성화 점검: APIM Diagnostics 의 custom metrics(metrics) 플래그
out, err, rc = arm("GET", "/diagnostics/applicationinsights", query="properties.metrics")
metrics_flag = (out or "").strip().strip('"').lower()
if metrics_flag != "true":
    print("  ⚠️ APIM Diagnostics 의 custom metrics 가 비활성(metrics != true) 입니다.")
    print("     → App Insights 'Custom metrics (Preview)' 활성화 + Diagnostics metrics:true 설정 후 재실행하세요.")
    print("     (자세한 절차: ../lab06-monitoring/README.md 2단계)\n")

if APP_INSIGHTS_APP_ID:
    print_table(query_app_insights(KQL_PER_SUB))
else:
    print("  APP_INSIGHTS_APP_ID 미설정 — 아래 저장된 KQL 을 Portal → Logs 에서 실행하세요.")


---
### 📋 KQL 쿼리 모음 — Portal(App Insights → Logs)에서 바로 실행

> 메트릭 이름: `Total Tokens` / `Prompt Tokens` / `Completion Tokens` (namespace `ai-gateway-metrics`)
> 구독 차원: `customDimensions["Subscription ID"]`

**1) 구독별 토큰 사용량 (최근 1시간)**
```kql
customMetrics
| where timestamp > ago(1h)
| where name in ("Total Tokens", "Prompt Tokens", "Completion Tokens")
| extend sub = tostring(customDimensions["Subscription ID"])
| summarize totalTokens = sumif(value, name=="Total Tokens"),
            promptTokens = sumif(value, name=="Prompt Tokens"),
            completionTokens = sumif(value, name=="Completion Tokens")
    by sub
| order by totalTokens desc
```

**2) 구독별 TPM(분당 토큰) 시계열**
```kql
customMetrics
| where timestamp > ago(1h)
| where name == "Total Tokens"
| extend sub = tostring(customDimensions["Subscription ID"])
| summarize TPM = sum(value) by sub, bin(timestamp, 1m)
| render timechart
```

**3) 구독별 quota 대비 사용률 (월 토큰 예산 대비)**
```kql
let quota = 2000000;   // team-b token-quota (프로덕션 값)
customMetrics
| where timestamp > ago(30d)
| where name == "Total Tokens"
| extend sub = tostring(customDimensions["Subscription ID"])
| summarize used = sum(value) by sub
| extend quota = quota, usedPct = round(100.0 * used / quota, 2)
```

**4) 429/403 거버넌스 차단 추이 (requests 테이블)**
```kql
requests
| where timestamp > ago(1h)
| where resultCode in ("429", "403")
| summarize count() by resultCode, bin(timestamp, 5m)
| render columnchart
```


---
## 정리: Product · 구독 삭제

테스트로 만든 `team-a` / `team-b` Product 와 구독을 삭제해 원상 복구합니다.


In [ ]:
# ─── 정리 ───
print("▶ 정리: 구독/Product 삭제\n")
for pid, s in SUBS.items():
    arm("DELETE", f"/subscriptions/{s['sid']}")
    print(f"  🧹 구독 삭제: {s['sid']}")
for pid in PRODUCTS:
    out, err, rc = az(["rest", "--method", "DELETE", "--url",
        f"{ARM_BASE}/products/{pid}?api-version={ARM_API}&deleteSubscriptions=true"])
    print(f"  🧹 Product 삭제: {pid}  {'✅' if rc==0 else err[:80]}")
print("\n✅ 정리 완료 — APIM 이 테스트 이전 상태로 복구되었습니다.")


---
## ✅ 요약: AI Gateway 의 제어 & 관측

| 레버 | 정책 | 무엇을 막나 | 응답 | 언제 쓰나 |
|---|---|---|---|---|
| 분당 토큰(TPM) | `llm-token-limit` `tokens-per-minute` | 순간 폭증 | **429** | 백엔드 보호, 공정 사용 |
| 누적 토큰 quota | `llm-token-limit` `token-quota` | 월 토큰 예산 | **403** | **비용(토큰) 상한** |
| 누적 요청 quota | `quota-by-key` `calls` | 월 호출 수 | **403** | 호출 수 상한, 남용 방지 |

**핵심**
- `counter-key = context.Subscription.Id` → **구독(=팀)별 독립 카운터**로 격리
- **토큰 레버 + 요청 레버는 상호 보완** — 함께 걸어 비용과 트래픽을 동시에 통제
- `llm-emit-token-metric` → App Insights `customMetrics` 로 **구독별 토큰 관측**
- Developer Portal 로 각 팀이 **셀프서비스 구독** → 키/토큰/메트릭이 subscriber 단위로 격리

→ 다음: [Lab 10 — 구독별 거버넌스 & Azure Monitor 대시보드](./README.md) · [Lab 10 폴더](../lab10-governance-observability/README.md)
